In [ ]:
import openeo

import glob
import os

from rasterio.mask import mask
from rasterio.merge import merge

import geopandas as gpd
import numpy as np

In [ ]:
from openeo.rest.auth.config import RefreshTokenStore
RefreshTokenStore().remove()

In [ ]:
url = "https://openeo.dataspace.copernicus.eu"
connection = openeo.connect(url).authenticate_oidc()

In [ ]:
spatial_extent_west = {"west": 610950, "east": 674650, "south": 5143820, "north": 5206380, "crs": "EPSG:32632"} # Western South Tyrol glaciers
spatial_extent_east = {"west": 704100, "east": 747040, "south": 5195880, "north": 5219660, "crs": "EPSG:32632"} # Eastern South Tyrol glaciers

In [ ]:
temporal_extent = ["2017-07-01T00:00:00Z", "2017-09-30T23:59:59Z"]

from pathlib import Path
output_folder = Path("")

In [ ]:
from pystac_client import Client
from datetime import datetime

catalog = Client.open("https://stac.dataspace.copernicus.eu/v1")

In [ ]:
# Search for Sentinel-1 GRD items
from pystac_client import Client
from pyproj import Transformer

catalog = Client.open("https://catalogue.dataspace.copernicus.eu/stac")

def utm_to_wgs84_bbox(extent):
    transformer = Transformer.from_crs(extent["crs"], "EPSG:4326", always_xy=True)
    west, south = transformer.transform(extent["west"], extent["south"])
    east, north = transformer.transform(extent["east"], extent["north"])
    return [west, south, east, north]

def get_orbits(bbox, datetime_range, label):
    search = catalog.search(
        collections=["sentinel-1-grd"],
        bbox=bbox,
        datetime=datetime_range,
    )
    items = sorted(search.items(), key=lambda i: i.datetime)
    print(f"\n{label}: {len(items)} scenes total")

    orbits = sorted(set(i.properties.get("sat:relative_orbit") for i in items))
    for o in orbits:
        dirs = set(i.properties.get("sat:orbit_state") for i in items if i.properties.get("sat:relative_orbit") == o)
        count = sum(1 for i in items if i.properties.get("sat:relative_orbit") == o)
        print(f"  orbit {o}: {dirs}, {count} scenes")
    return items

datetime_range = f"{temporal_extent[0]}/{temporal_extent[1]}"

items_west = get_orbits(utm_to_wgs84_bbox(spatial_extent_west), datetime_range, "West")
items_east = get_orbits(utm_to_wgs84_bbox(spatial_extent_east), datetime_range, "East")

In [ ]:
orbit_groups = {
    "west": [(15, "ascending"), (95, "descending"), (117, "ascending"), (168, "descending")],
    "east": [(44, "ascending"), (95, "descending"), (117, "ascending"), (168, "descending")],
}

In [ ]:
from openeo.processes import eq

def load_s1_coherence_stack(spatial_extent, relative_orbit, orbit_state, connection, temporal_extent):
    return connection.load_collection(
        "SENTINEL1_GRD",  
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
        bands=["VV", "VH", "HH"],
        properties={
            "sat:relative_orbit": lambda x, o=relative_orbit: eq(x, o),
            "sat:orbit_state": lambda x, s=orbit_state: eq(x, s),
        }
    )

west_cubes = {
    (orbit, direction): load_s1_coherence_stack(spatial_extent_west, orbit, direction, connection, temporal_extent)
    for orbit, direction in orbit_groups["west"]
}
east_cubes = {
    (orbit, direction): load_s1_coherence_stack(spatial_extent_east, orbit, direction, connection, temporal_extent)
    for orbit, direction in orbit_groups["east"]
}

In [ ]:
def reduce_and_save(cubes, reducer, out_dir, region_label):
    out_dir.mkdir(parents=True, exist_ok=True)
    for (orbit, direction), cube in cubes.items():
        reduced = cube.reduce_dimension(dimension="t", reducer=reducer)
        job = reduced.execute_batch(
            outputfile=out_dir / f"{region_label}_orbit{orbit}_{direction}_{reducer}.tif",
            title=f"S1_{region_label}_orbit{orbit}_{direction}_{reducer}"
        )

reduce_and_save(west_cubes, "median", output_folder / "s1_backscatter", "west")
reduce_and_save(east_cubes, "median", output_folder / "s1_backscatter", "east")

In [ ]:
# apply speckle filtering to the pre & the post image before calculating the ratio

import numpy as np
from scipy.ndimage import uniform_filter


def lee_filter(img, size=7):
    img = img.astype(np.float32)

    mean = uniform_filter(img, size)
    mean_sq = uniform_filter(img**2, size)

    variance = mean_sq - mean**2

    # estimate noise variance
    noise_var = np.mean(variance)

    weights = variance / (variance + noise_var)

    filtered = mean + weights * (img - mean)

    return filtered

In [ ]:
with rasterio.open("") as src:
    img_pre = src.read()
    profile = src.profile
    arr = src.read(1)
    print(arr.min(), arr.max(), arr.mean())

#filtered_pre = lee_filter(img_pre, size=5)

#with rasterio.open("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/flood/pre_cube_s1_median_filtered.tif", "w", **profile) as dst:
#    dst.write(filtered_pre)